In [2]:
from SPARQLWrapper import SPARQLWrapper, JSON
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd
from rdflib import Graph, URIRef, RDFS
import csv
import re
from collections import Counter, defaultdict
import csv
import re
import requests
import time
import os
import urllib.parse
from bs4 import BeautifulSoup

In [71]:
#pip freeze > requirements.txt

In [3]:
# Set up the endpoint
endpoint_url = "https://query.wikidata.org/sparql"
sparql = SPARQLWrapper(endpoint_url)
sparql.setReturnFormat(JSON)

### Direct connections to instances of Holy Well/ Holy Well Semantic Concept

In [4]:
# Get full triples for holy wells semantic concept
query = """
SELECT ?subject ?predicate ?object WHERE {
#holy well semantic concepts alone
  ?subject wdt:P31 wd:Q126443332 .
  ?subject ?predicate ?object .
}
"""
sparql.setQuery(query)

# Execute
results = sparql.query().convert()

# Convert to dataframe
triples = [
    {
        "subject": result["subject"]["value"],
        "predicate": result["predicate"]["value"],
        "object": result["object"]["value"]
    }
    for result in results["results"]["bindings"]
]

df_triples = pd.DataFrame(triples)
df_triples.head()
df_triples = df_triples[df_triples["predicate"].str.contains("/direct/")]
df_triples.to_csv("holy_wells_semantic_concept.csv", index=False)


In [5]:
def get_labels(codes):
    """Fetch labels for a list of Q or P codes from Wikidata."""
    labels = {}
    endpoint = "https://query.wikidata.org/sparql"
    batch_size = 50

    for i in range(0, len(codes), batch_size):
        chunk = codes[i:i + batch_size]
        uris = " ".join(f"wd:{c}" for c in chunk)
        query = f"""
        SELECT ?code ?codeLabel WHERE {{
          VALUES ?code {{{uris}}}
          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
        }}
        """
        headers = {
            "Accept": "application/sparql-results+json"
        }

        try:
            response = requests.get(endpoint, params={'query': query}, headers=headers, timeout=30)
            response.raise_for_status()
            data = response.json()
            for result in data['results']['bindings']:
                code = result['code']['value'].split('/')[-1]
                label = result['codeLabel']['value']
                labels[code] = label
        except requests.exceptions.RequestException as e:
            print(f"HTTP request failed: {e}")
            print("Query:")
            print(query)
            continue
        except ValueError:
            print("Response was not valid JSON:")
            print(response.text[:500])
            continue

        time.sleep(1)  # Respect rate limits

    return labels


In [4]:
def extract_labels_from_csv(csv_file_path, limit=None):
    import csv
    import os
    import re

    base_name = os.path.splitext(os.path.basename(csv_file_path))[0]
    qcode_output = f'qcode_labels_{base_name}.csv'
    pcode_output = f'pcode_labels_{base_name}.csv'

    q_pattern = re.compile(r'Q\d+')
    p_pattern = re.compile(r'(P\d+)')
    qcodes, pcodes = set(), set()

    with open(csv_file_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            subj_match = q_pattern.search(row['subject'])
            obj_match = q_pattern.search(row['object'])
            pred_match = p_pattern.search(row['predicate'])

            if subj_match: qcodes.add(subj_match.group())
            if obj_match: qcodes.add(obj_match.group())
            if pred_match: pcodes.add(pred_match.group())

    qcodes = list(qcodes)[:limit] if limit else list(qcodes)
    pcodes = list(pcodes)[:limit] if limit else list(pcodes)

    # Use shared get_labels
    qcode_labels = get_labels(qcodes)
    pcode_labels = get_labels(pcodes)

    def save_dict_to_csv(filename, data_dict, header):
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(header)
            for k, v in sorted(data_dict.items()):
                writer.writerow([k, v])

    save_dict_to_csv(qcode_output, qcode_labels, ['Q-code', 'Label'])
    save_dict_to_csv(pcode_output, pcode_labels, ['P-code', 'Label'])

    print(f"Saved {len(qcode_labels)} Q-code labels to {qcode_output}")
    print(f"Saved {len(pcode_labels)} P-code labels to {pcode_output}")


In [9]:
extract_labels_from_csv("holy_wells_semantic_concept.csv")

Saved 787 Q-code labels to qcode_labels_holy_wells_semantic_concept.csv
Saved 37 P-code labels to pcode_labels_holy_wells_semantic_concept.csv


In [ ]:
# query to get en label of property
# SELECT ?property ?label WHERE {
# BIND(wd:P31 AS ?property)
# ?property rdfs:label ?label .
# FILTER(LANG(?label) = "en")}

### Stats for instance of Holy Well Semantic Concept

In [6]:
#Get entity with most unique direct predicates -columbkile's well
entity_query = """
SELECT ?item (COUNT(DISTINCT ?prop) AS ?propCount) WHERE {
  ?item wdt:P31 wd:Q126443332 .
  ?item ?prop ?val .
}
GROUP BY ?item
ORDER BY DESC(?propCount)
LIMIT 1
"""

sparql.setQuery(entity_query)
results_most_unique_predicates = sparql.query().convert()
top_entity = results_most_unique_predicates["results"]["bindings"][0]["item"]["value"]

print(f"Entity with most unique predicates: {top_entity}")


Entity with most unique predicates: http://www.wikidata.org/entity/Q126456441


In [11]:
#most to least used direct predicates
csv_file = 'holy_wells_semantic_concept.csv'
pcode_labels_file = 'pcode_labels_holy_wells_semantic_concept.csv'  # already extracted

# Step 1: Load P-code → label mapping
pcode_to_label = {}
with open(pcode_labels_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        pcode_to_label[row['P-code']] = row['Label']

# Step 2: Count P-code usage in the main CSV
pcode_counter = Counter()
pattern = re.compile(r'/direct/(P\d+)')

with open(csv_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        match = pattern.search(row['predicate'])
        if match:
            pcode = match.group(1)
            pcode_counter[pcode] += 1

# Step 3: Output results with label
print(f"{'P-code':<8} {'Count':<6} Label")
print("-" * 40)
for pcode, count in pcode_counter.most_common():
    label = pcode_to_label.get(pcode, "[Label not found]")
    print(f"{pcode:<8} {count:<6} {label}")


P-code   Count  Label
----------------------------------------
P1343    1742   described by source
P31      560    instance of
P131     557    located in the administrative territorial entity
P217     379    inventory number
P195     374    collection
P17      229    country
P625     213    coordinate location
P138     201    named after
P708     192    diocese
P4057    169    Irish Sites and Monuments Record ID
P11693   140    OpenStreetMap node ID
P18      86     image
P6375    77     street address
P373     70     Commons category
P973     46     described at URL
P841     36     feast day
P1325    33     external data available at URL
P2186    33     Wiki Loves Monuments ID
P10      20     video
P1651    19     YouTube video ID
P403     8      mouth of the watercourse
P2175    6      medical condition treated
P10689   6      OpenStreetMap way ID
P110     5      illustrator
P7959    5      historic county
P1050    3      medical condition
P4088    3      Irish National Inventory of A

In [12]:
#todo: P51      1      audio - works like video or image but also really?? 
#todo: P2441    1      literal translation
#todo: P571     1      inception
#todo: P1801    1      plaque image -- difference to image??

In [ ]:
#connection url: jdbc:postgresql://localhost:5432/ontopdb
#user: ontopuser
#pwd: 1234
#driver: org.postgresql.driver

#### Columbkille's Well

In [7]:
target_qcode = "Q126456441" # columbkille's well
csv_file = 'holy_wells_semantic_concept.csv'
qcode_labels_file = 'qcode_labels_holy_wells_semantic_concept.csv'
pcode_labels_file = 'pcode_labels_holy_wells_semantic_concept.csv'

In [14]:
# Load Q-code labels
qcode_to_label = {}
with open(qcode_labels_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        qcode_to_label[row['Q-code']] = row['Label']

# Load P-code labels
pcode_to_label = {}
with open(pcode_labels_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        pcode_to_label[row['P-code']] = row['Label']

# Regex patterns
q_pattern = re.compile(r'Q\d+')
p_pattern = re.compile(r'/direct/(P\d+)')

# Process triples
print(f"{'Subject':<30} {'Predicate':<40} {'Object'}")
print("-" * 100)

with open(csv_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        subj_match = q_pattern.search(row['subject'])
        obj_match = q_pattern.search(row['object'])
        pred_match = p_pattern.search(row['predicate'])

        subj_q = subj_match.group(0) if subj_match else None
        obj_q = obj_match.group(0) if obj_match else None
        pcode = pred_match.group(1) if pred_match else None

        if target_qcode in (subj_q, obj_q):
            subj_label = qcode_to_label.get(subj_q, subj_q or "[Not Q-code]")
            pred_label = pcode_to_label.get(pcode, pcode or "[Not P-code]")
            obj_label = qcode_to_label.get(obj_q, obj_q or row['object'])

            print(f"{subj_label:<30} {pred_label:<40} {obj_label}")


Subject                        Predicate                                Object
----------------------------------------------------------------------------------------------------
Columbkille's Well             video                                    http://commons.wikimedia.org/wiki/Special:FilePath/St%20Colombkille%27s%20Well.webm
Columbkille's Well             country                                  Ireland
Columbkille's Well             image                                    http://commons.wikimedia.org/wiki/Special:FilePath/Inistioge%20Saint%20Columbkille%27s%20Well.png
Columbkille's Well             image                                    http://commons.wikimedia.org/wiki/Special:FilePath/Water%20source%20at%20Columbkille%27s%20Well.jpg
Columbkille's Well             image                                    http://commons.wikimedia.org/wiki/Special:FilePath/Water%20tap%20from%20Columbkille%27s%20Well.jpg
Columbkille's Well             instance of                             

## Functions to fetch a star around a subject

### Instance of

In [18]:
def get_instance_of(qid: str) -> pd.DataFrame:
    """
    Fetch all 'instance of' (P31) statements for a given Wikidata entity,
    returning subject Q, instance-of Q + label, and stated-in Q + label (if present).
    """

    query = f"""
    SELECT ?instanceOf ?instanceOfLabel ?statedIn ?statedInLabel
    WHERE {{
      wd:{qid} p:P31 ?stmt .
      ?stmt ps:P31 ?instanceOf .

      OPTIONAL {{
        ?instanceOf rdfs:label ?instanceOfLabel .
        FILTER(LANG(?instanceOfLabel) = "en")
      }}

      OPTIONAL {{
        ?stmt prov:wasDerivedFrom ?ref .
        ?ref pr:P248 ?statedIn .
        OPTIONAL {{
          ?statedIn rdfs:label ?statedInLabel .
          FILTER(LANG(?statedInLabel) = "en")
        }}
      }}
    }}
    """

    sparql.setQuery(query)
    results = sparql.query().convert()

    rows = []
    for b in results["results"]["bindings"]:
        rows.append({
            "subject_q": qid,
            "instanceOf_q": b.get("instanceOf", {}).get("value", "").split("/")[-1],
            "instanceOf": b.get("instanceOfLabel", {}).get("value", ""),
            "stated_in_q": b.get("statedIn", {}).get("value", "").split("/")[-1] if "statedIn" in b else "",
            "stated_in": b.get("statedInLabel", {}).get("value", ""),
        })

    return pd.DataFrame(rows)




In [21]:
df = get_instance_of("Q126456441")
df.to_csv("instance_of.csv", index=False)

### Description, aliases, language

In [ ]:
#get description, aliases, language - modelled in protege
def fetch_description_alias_language(subject_qcode: str) -> pd.DataFrame:
    query = f"""
    SELECT ?lang ?valueType ?value WHERE {{
      VALUES ?item {{ wd:{subject_qcode} }}
      {{
        ?item rdfs:label ?value .
        BIND("label" AS ?valueType)
        BIND(LANG(?value) AS ?lang)
      }}
      UNION
      {{
        ?item schema:description ?value .
        BIND("description" AS ?valueType)
        BIND(LANG(?value) AS ?lang)
      }}
      UNION
      {{
        ?item skos:altLabel ?value .
        BIND("alias" AS ?valueType)
        BIND(LANG(?value) AS ?lang)
      }}
    }}
    ORDER BY ?lang ?valueType
    """
    sparql.setQuery(query)
    results = sparql.query().convert()
    
    # Collect descriptions and aliases per language
    data = defaultdict(lambda: {"description": "", "aliases": []})
    for binding in results["results"]["bindings"]:
        lang = binding["lang"]["value"]
        vtype = binding["valueType"]["value"]
        val   = binding["value"]["value"]
        if vtype == "description":
            data[lang]["description"] = val
        elif vtype == "alias":
            data[lang]["aliases"].append(val)
    
    # Build final rows
    rows = []
    for lang, content in data.items():
        desc = content["description"]
        if content["aliases"]:
            for alias in content["aliases"]:
                rows.append({
                    "subject": subject_qcode,
                    "language": lang,
                    "description": desc,
                    "alias": alias
                })
        else:
            rows.append({
                "subject_q": subject_qcode,
                "language": lang,
                "description": desc,
                "alias": ""
            })
    
    return pd.DataFrame(rows, columns=["subject", "language", "description", "alias"])



In [ ]:
df = fetch_description_alias_language(target_qcode)
df.to_csv("lang_description_aliases.csv", index=False)


### statements and references

In [ ]:
#todo: references, mentions for what is left. check for start-end dates for everything that is a date. what else is there to do?

In [ ]:
#illustrator, work and their instance of - modelled in protege. could get the meta for the monograph too.
def get_illustrator_references(subject_q):
    query = f"""
    SELECT ?illustrator ?illustratorLabel 
           ?illustratorInstance ?illustratorInstanceLabel
           ?statedIn ?statedInLabel 
           ?statedInInstanceOf ?statedInInstanceOfLabel 
    WHERE {{
      wd:{subject_q} p:P110 ?statement .
      ?statement ps:P110 ?illustrator .

      OPTIONAL {{
        ?illustrator wdt:P31 ?illustratorInstance .
      }}

      OPTIONAL {{
        ?statement prov:wasDerivedFrom ?ref .
        ?ref pr:P248 ?statedIn .
        
        OPTIONAL {{
          ?statedIn p:P31 ?instanceStatement .
          ?instanceStatement ps:P31 ?statedInInstanceOf .
        }}
      }}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()

    records = []
    for b in results["results"]["bindings"]:
        records.append({
            "subject_q": subject_q,
            "illustrator_q": b["illustrator"]["value"].split("/")[-1],
            "illustrator_label": b.get("illustratorLabel", {}).get("value", ""),
            "illustrator_instance_q": b.get("illustratorInstance", {}).get("value", "").split("/")[-1] if "illustratorInstance" in b else "",
            "illustrator_instance_label": b.get("illustratorInstanceLabel", {}).get("value", "") if "illustratorInstanceLabel" in b else "",
            "stated_in_q": b.get("statedIn", {}).get("value", "").split("/")[-1] if "statedIn" in b else "",
            "stated_in_label": b.get("statedInLabel", {}).get("value", "") if "statedInLabel" in b else "",
            "stated_in_instance_q": b.get("statedInInstanceOf", {}).get("value", "").split("/")[-1] if "statedInInstanceOf" in b else "",
            "stated_in_instance_of_label": b.get("statedInInstanceOfLabel", {}).get("value", "") if "statedInInstanceOfLabel" in b else "",
        })

    return pd.DataFrame(records)


In [ ]:
#Toberbride as example
df = get_illustrator_references("Q122222699")
df.to_csv("illustrator.csv", index=False)

In [ ]:
# state and metadata - modeled in protege
def fetch_state_qualifiers_with_source_metadata(well_qid):
    from SPARQLWrapper import SPARQLWrapper, JSON
    import pandas as pd

    # Main query
    query = f"""
    SELECT ?qualifier_property ?qualifier_value ?label ?description 
           ?stated_in_qid ?stated_in_label ?source_prop ?source_value ?source_value_label WHERE {{
      BIND(wd:{well_qid} AS ?well)

      ?well p:P31 ?stmt .
      ?stmt ?pq ?qualifier_value .
      FILTER(?pq IN (pq:P5817, pq:P5816))  # state of use, state of conservation

      BIND(STR(REPLACE(STR(?pq), "^.*(P\\\\d+)$", "$1")) AS ?qualifier_property)

      OPTIONAL {{
        ?qualifier_value rdfs:label ?label .
        FILTER(LANG(?label) = "en")
      }}

      OPTIONAL {{
        ?qualifier_value schema:description ?description .
        FILTER(LANG(?description) = "en")
      }}

      OPTIONAL {{
        ?stmt prov:wasDerivedFrom ?ref .
        ?ref pr:P248 ?stated_in .
        BIND(STRAFTER(STR(?stated_in), "entity/") AS ?stated_in_qid)

        OPTIONAL {{
          ?stated_in rdfs:label ?stated_in_label .
          FILTER(LANG(?stated_in_label) = "en")
        }}

        OPTIONAL {{
          ?stated_in ?source_pred_uri ?source_value .
          FILTER(STRSTARTS(STR(?source_pred_uri), "http://www.wikidata.org/prop/direct/"))
          BIND(STRAFTER(STR(?source_pred_uri), "/prop/direct/") AS ?source_prop)

          OPTIONAL {{
            ?source_value rdfs:label ?source_value_label .
            FILTER(LANG(?source_value_label) = "en")
          }}
        }}
      }}
    }}
    """

    endpoint_url = "https://query.wikidata.org/sparql"
    sparql = SPARQLWrapper(endpoint_url)
    sparql.setReturnFormat(JSON)
    sparql.setQuery(query)
    results = sparql.query().convert()

    rows = {}
    all_source_props = set()
    all_source_qid_props = set()

    for b in results["results"]["bindings"]:
        key = (
            well_qid,
            b.get("qualifier_property", {}).get("value", ""),
            b.get("qualifier_value", {}).get("value", ""),
            b.get("stated_in_qid", {}).get("value", "")
        )

        if key not in rows:
            rows[key] = {
                "subject_q": well_qid,
                "predicate_p": b.get("qualifier_property", {}).get("value", ""),
                "object_q": b.get("qualifier_value", {}).get("value", "").split("/")[-1],
                "object": b.get("label", {}).get("value", "") or b.get("description", {}).get("value", ""),
                "stated_in_q": b.get("stated_in_qid", {}).get("value", ""),
                "stated_in_label": b.get("stated_in_label", {}).get("value", "")
            }

        source_prop = b.get("source_prop", {}).get("value")
        source_value_uri = b.get("source_value", {}).get("value")
        source_value_label = b.get("source_value_label", {}).get("value")

        if source_prop:
            # Add label
            prop_value = source_value_label or source_value_uri
            if prop_value:
                rows[key][source_prop] = prop_value
                all_source_props.add(source_prop)

            # Add QID
            if source_value_uri and "entity/" in source_value_uri:
                qid = source_value_uri.split("/")[-1]
                qid_key = f"{source_prop}_q"
                rows[key][qid_key] = qid
                all_source_qid_props.add(qid_key)

    # Fetch property labels
    all_p_codes = all_source_props.union(p.replace("_q", "") for p in all_source_qid_props)
    prop_labels = {}

    for p_code in all_p_codes:
        sparql.setQuery(f"""
        SELECT ?label WHERE {{
          wd:{p_code} rdfs:label ?label .
          FILTER(LANG(?label) = "en")
        }} LIMIT 1
        """)
        try:
            res = sparql.query().convert()
            label = res["results"]["bindings"][0]["label"]["value"]
            prop_labels[p_code] = label
        except Exception:
            prop_labels[p_code] = p_code  # fallback to P-code if label not found

    # Build final column headers
    standard_headers = ["subject_q", "predicate_p", "object_q", "object", "stated_in_q", "stated_in_label"]

    grouped_source_headers = []
    for p_code in sorted(all_p_codes):
        label = prop_labels[p_code]
        grouped_source_headers.extend([label, f"{label}_q"])

    all_headers = standard_headers + grouped_source_headers

    # Normalize rows
    normalized_rows = []
    for row in rows.values():
        norm_row = {h: "" for h in all_headers}
        for h in standard_headers:
            norm_row[h] = row.get(h, "")

        for p_code in all_p_codes:
            label = prop_labels[p_code]
            label_col = label
            qid_col = f"{label}_q"

            norm_row[label_col] = row.get(p_code, "")
            norm_row[qid_col] = row.get(f"{p_code}_q", "")

        normalized_rows.append(norm_row)

    return pd.DataFrame(normalized_rows)


In [25]:
# Toberbride as example
df = fetch_state_qualifiers_with_source_metadata("Q122258940")
df.to_csv("state.csv", index=False)

In [ ]:
#street address 
def get_street_address_references(subject_q):
    query = f"""
    SELECT DISTINCT ?streetText ?statedIn ?statedInLabel ?statedInInstanceOf ?statedInInstanceOfLabel WHERE {{
      wd:{subject_q} p:P6375 ?statement .
      ?statement ps:P6375 ?streetText .

      OPTIONAL {{
        ?statement prov:wasDerivedFrom ?ref .
        ?ref pr:P248 ?statedIn .

        OPTIONAL {{
          ?statedIn p:P31 ?instanceStatement .
          ?instanceStatement ps:P31 ?statedInInstanceOf .
        }}
      }}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()

    records = []
    for b in results["results"]["bindings"]:
        records.append({
            "subject_q": subject_q,
            "street_address": b.get("streetText", {}).get("value", ""),
            "stated_in_q": b.get("statedIn", {}).get("value", "").split("/")[-1] if "statedIn" in b else "",
            "stated_in_label": b.get("statedInLabel", {}).get("value", "") if "statedInLabel" in b else "",
            "stated_in_instance_q": b.get("statedInInstanceOf", {}).get("value", "").split("/")[-1] if "statedInInstanceOf" in b else "",
            "stated_in_instance_label": b.get("statedInInstanceOfLabel", {}).get("value", "") if "statedInInstanceOfLabel" in b else "",
        })

    return pd.DataFrame(records)


In [ ]:
# St. Bridget's Well as example
df = get_street_address_references("Q126454471")
df.to_csv("street_addr.csv", index=False)

#### URL

In [ ]:
# modelled in protege - add a column with a copy of the url with all / replaced with _ 
def get_url_meta(subject_q, predicate_p):
    query = f"""
    PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
    PREFIX p: <http://www.wikidata.org/prop/>
    PREFIX ps: <http://www.wikidata.org/prop/statement/>
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?url ?qualifierProp ?qualifierPropLabel ?qualifierValue ?qualifierValueLabel ?qualifierValueP31 ?qualifierValueP31Label WHERE {{
      wd:{subject_q} p:{predicate_p} ?statement .
      ?statement ps:{predicate_p} ?url .

      OPTIONAL {{
        ?statement ?qualifierProp ?qualifierValue .
        FILTER(STRSTARTS(STR(?qualifierProp), STR(pq:)))

        # Get label of qualifier property
        BIND(IRI(CONCAT("http://www.wikidata.org/entity/",
            STRAFTER(STR(?qualifierProp), "http://www.wikidata.org/prop/qualifier/"))) AS ?propEntity)
        ?propEntity rdfs:label ?qualifierPropLabel .
        FILTER(LANG(?qualifierPropLabel) = "en")

        # If qualifierValue is an entity, get its P31 (instance of) and label
        OPTIONAL {{
          FILTER(STRSTARTS(STR(?qualifierValue), STR(wd:)))
          ?qualifierValue wdt:P31 ?qualifierValueP31 .
          ?qualifierValueP31 rdfs:label ?qualifierValueP31Label .
          FILTER(LANG(?qualifierValueP31Label) = "en")
        }}
      }}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()

    rows = []
    for res in results["results"]["bindings"]:
        url = res["url"]["value"]
        qualifierPropLabel = res.get("qualifierPropLabel", {}).get("value")
        qualifierValueLabel = res.get("qualifierValueLabel", {}).get("value")
        qualifierValueRaw = res.get("qualifierValue", {}).get("value")
        
        # Prefer the label of the qualifier value, fallback to raw value (URI or literal)
        qualifierValue = qualifierValueLabel if qualifierValueLabel else qualifierValueRaw
        
        qualifierValueP31 = res.get("qualifierValueP31", {}).get("value")
        # Extract Q code from the full URI, if present
        if qualifierValueP31:
            qualifierValueP31 = qualifierValueP31.split("/")[-1]
        qualifierValueP31Label = res.get("qualifierValueP31Label", {}).get("value")

        rows.append({
            "subject_q":subject_q,
            "url": url,
            "qualifier_property": qualifierPropLabel,
            "qualifier_value": qualifierValue,
            "qualifier_value_p31": qualifierValueP31,
            "qualifier_value_p31_label": qualifierValueP31Label
        })
    df = pd.DataFrame(rows)
    return df


In [26]:
def get_url_meta(subject_q, predicate_p):
    query = f"""
    PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
    PREFIX p: <http://www.wikidata.org/prop/>
    PREFIX ps: <http://www.wikidata.org/prop/statement/>
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?url ?qualifierProp ?qualifierPropLabel ?qualifierValue ?qualifierValueLabel ?qualifierValueP31 ?qualifierValueP31Label WHERE {{
      wd:{subject_q} p:{predicate_p} ?statement .
      ?statement ps:{predicate_p} ?url .

      OPTIONAL {{
        ?statement ?qualifierProp ?qualifierValue .
        FILTER(STRSTARTS(STR(?qualifierProp), STR(pq:)))

        # Get label of qualifier property
        BIND(IRI(CONCAT("http://www.wikidata.org/entity/",
            STRAFTER(STR(?qualifierProp), "http://www.wikidata.org/prop/qualifier/"))) AS ?propEntity)
        ?propEntity rdfs:label ?qualifierPropLabel .
        FILTER(LANG(?qualifierPropLabel) = "en")

        # If qualifierValue is an entity, get its P31 (instance of) and label
        OPTIONAL {{
          FILTER(STRSTARTS(STR(?qualifierValue), STR(wd:)))
          ?qualifierValue wdt:P31 ?qualifierValueP31 .
          ?qualifierValueP31 rdfs:label ?qualifierValueP31Label .
          FILTER(LANG(?qualifierValueP31Label) = "en")
        }}
      }}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()

    rows = []
    for res in results["results"]["bindings"]:
        url = res["url"]["value"]
        qualifierPropLabel = res.get("qualifierPropLabel", {}).get("value")
        qualifierValueLabel = res.get("qualifierValueLabel", {}).get("value")
        qualifierValueRaw = res.get("qualifierValue", {}).get("value")
        
        # Prefer the label of the qualifier value, fallback to raw value (URI or literal)
        qualifierValue = qualifierValueLabel if qualifierValueLabel else qualifierValueRaw
        
        qualifierValueP31 = res.get("qualifierValueP31", {}).get("value")
        # Extract Q code from the full URI, if present
        if qualifierValueP31:
            qualifierValueP31 = qualifierValueP31.split("/")[-1]
        qualifierValueP31Label = res.get("qualifierValueP31Label", {}).get("value")

        # Add new column with URL where all '/' replaced with '_'
        url_underscore = url.replace("/", "_")

        rows.append({
            "subject_q": subject_q,
            "url": url,
            "url_underscore": url_underscore,
            "qualifier_property": qualifierPropLabel,
            "qualifier_value": qualifierValue,
            "qualifier_value_p31": qualifierValueP31,
            "qualifier_value_p31_label": qualifierValueP31Label
        })
    df = pd.DataFrame(rows)
    return df


In [29]:
#described at URL
df = get_url_meta("Q114439798", "P973")
df.to_csv("url_meta.csv", index=False)

In [31]:
#external data available
df = get_url_meta("Q114439798", "P1325")
df.to_csv("url_external.csv", index=False)

In [33]:
# non-free artwork image URL
df = get_url_meta("Q126472997", "P6500")
df.to_csv("url_nonfree.csv", index=False)

#### Inventory-Collection

In [ ]:
#inventory number with collection - modeled in protege
def fetch_well_inventory(target_qcode: str) -> pd.DataFrame:
    query = f"""
    SELECT ?well ?invNum ?collection ?label WHERE {{
      BIND(wd:{target_qcode} AS ?well)

      ?well wdt:P31 wd:Q126443332 ;       # Ensure it's a holy well semantic concept
            p:P217 ?stmt .                # Get inventory number statement node

      ?stmt ps:P217 ?invNum .             # The inventory number value
      ?stmt pq:P195 ?collection .         # The associated collection (qualifier)

      OPTIONAL {{
        ?collection rdfs:label ?label .   # Collection label
        FILTER (lang(?label) = "en")
      }}
    }}
    """

    sparql.setQuery(query)
    results = sparql.query().convert()
    
    # Parse bindings into rows
    rows = [
        {
            "subject_q": target_qcode,
            "inventory_id": binding["invNum"]["value"],
            "collection_q": binding["collection"]["value"].rsplit("/", 1)[-1],
            "collection_label": binding.get("label", {}).get("value", "")
        }
        for binding in results["results"]["bindings"]
    ]

    # Create DataFrame
    df = pd.DataFrame(rows)
    return df


In [ ]:
df = fetch_well_inventory(target_qcode)
save_df(df, "inventory_id_coll.csv")


#### YT Video and metadata

In [4]:
#YouTube video ID, subject named as, nr of views, publication date, duration, yt channel id, point in time
#modeled in protege
def fetch_well_yt_videos(target_qcode: str) -> pd.DataFrame:
    query = f"""
    SELECT ?vidID ?date ?dur WHERE {{
      wd:{target_qcode} p:P1651 ?stmt .
      ?stmt ps:P1651 ?vidID .
      OPTIONAL {{ ?stmt pq:P577  ?date }}
      OPTIONAL {{ ?stmt pq:P2047 ?dur  }}
    }}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()

    rows = []
    for b in results["results"]["bindings"]:
        rows.append({
            "subject_q":          target_qcode,
            "youtube_id":        b["vidID"]["value"],
            "publication_date":  b.get("date", {}).get("value"),
            "duration":          b.get("dur",  {}).get("value"),
        })

    return pd.DataFrame(rows, columns=[
        "subject_q", "youtube_id", "publication_date", "duration"
    ])


In [8]:
df = fetch_well_yt_videos(target_qcode)
df.to_csv("YouTube_Video_ID_Metadata.csv", index=False)
print(df)

    subject_q   youtube_id      publication_date duration
0  Q126456441  kxRnXwX0HAM  2024-05-19T00:00:00Z      131


#### Image and Video metadata from wiki commons

In [76]:
def extract_name_and_link(html_string):
    if not html_string:
        return "", ""
    soup = BeautifulSoup(html_string, "html.parser")
    tag = soup.find("a")
    if tag:
        return tag.get_text(strip=True), f"https:{tag['href']}" if tag['href'].startswith("//") else tag['href']
    else:
        return soup.get_text(strip=True), ""


In [ ]:
def get_meta(subject_q, predicate_p):
    # Step 1: SPARQL to get all images
    query = f"""
    SELECT ?image WHERE {{
      wd:{subject_q} wdt:{predicate_p} ?image .
    }}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()
    bindings = results["results"]["bindings"]

    if not bindings:
        return pd.DataFrame(columns=["Q", "file_url", "commons_file_url", "wikimedia_url"])

    rows = []

    for binding in bindings:
        image_url = binding["image"]["value"]
        filename = image_url.split("/")[-1]
        decoded_filename = unquote(filename).replace(" ", "_")
        commons_filename = f"File:{decoded_filename}"

        # Step 2: Query Wikimedia API
        api_url = "https://commons.wikimedia.org/w/api.php"
        params = {
            "action": "query",
            "prop": "imageinfo",
            "titles": commons_filename,
            "iiprop": "url|extmetadata",
            "format": "json",
        }

        response = requests.get(api_url, params=params)
        data = response.json()

        # Step 3: Parse metadata
        pages = data.get("query", {}).get("pages", {})
        for page_id, page_data in pages.items():
            imageinfo = page_data.get("imageinfo", [{}])[0]
            wikimedia_url = imageinfo.get("url", "")
            extmetadata = imageinfo.get("extmetadata", {})

            row = {
                "Q": subject_q,
                "file_url": image_url,
                "commons_file_url": f"https://commons.wikimedia.org/wiki/{commons_filename}",
                "wikimedia_url": wikimedia_url
            }

            for key, meta in extmetadata.items():
                if key != "Categories":  # Exclude Categories
                    row[key] = meta.get("value", "")

            # Clean and split Artist
            artist_html = extmetadata.get("Artist", {}).get("value", "")
            artist_name, artist_link = extract_name_and_link(artist_html)
            row["artist_name"] = artist_name
            row["artist_link"] = artist_link

            # Clean and split Credit
            credit_html = extmetadata.get("Credit", {}).get("value", "")
            credit_name, credit_link = extract_name_and_link(credit_html)
            row["credit_name"] = credit_name
            row["credit_link"] = credit_link

            # Remove raw Artist and Credit
            row.pop("Artist", None)
            row.pop("Credit", None)
            row.pop("Categories", None)

            rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
#image and metadata
df = get_meta(target_qcode, "P18")
df.to_csv("image_meta.csv", index=False)

In [ ]:
#video and metadata
df = get_meta(target_qcode, "P10")
df.to_csv("video_meta.csv", index=False)

### direct connections - done in protege

#### Coordinates and point

In [105]:
def dms_string(lat, lon):
    def to_dms(deg):
        d = int(deg)
        m_float = abs((deg - d) * 60)
        m = int(m_float)
        s = round((m_float - m) * 60, 4)
        return d, m, s

    lat_d, lat_m, lat_s = to_dms(abs(lat))
    lon_d, lon_m, lon_s = to_dms(abs(lon))

    lat_hem = "N" if lat >= 0 else "S"
    lon_hem = "E" if lon >= 0 else "W"

    lat_str = f'{lat_d}°{lat_m}\'{lat_s}"{lat_hem}'
    lon_str = f'{lon_d}°{lon_m}\'{lon_s}"{lon_hem}'
    return f"{lat_str}, {lon_str}"

In [107]:
def get_well_coordinates(qcode):
    query = f"""
    SELECT ?coord WHERE {{
      wd:{qcode} wdt:P625 ?coord .
    }}
    """

    sparql.setQuery(query)
    results = sparql.query().convert()
    bindings = results["results"]["bindings"]

    if not bindings:
        return None, None

    wkt = bindings[0]["coord"]["value"]  # "Point(lon lat)"
    try:
        lon_str, lat_str = wkt.replace("Point(", "").replace(")", "").split()
        lat, lon = float(lat_str), float(lon_str)
        dms = dms_string(lat, lon)
        return wkt, dms
    except:
        return None, None

def save_coordinates_to_csv(qcode, filename):
    point, dms = get_well_coordinates(qcode)
    if not point or not dms:
        print(f"No coordinates found for {qcode}")
        return

    file_exists = os.path.isfile(filename)
    with open(filename, mode="a", newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["subject_q", "object-point", "object-coords"])
        writer.writerow([qcode, point, dms])
    print(f"Saved: {qcode}, {point}, {dms}")

In [ ]:
# well coordinates - modelled in protege
save_coordinates_to_csv(target_qcode, "coordinates.csv")

Saved: Q126456441, Point(-7.0681791 52.4892388), 52°29'21.2597"N, 7°4'5.4448"W


In [ ]:
#mouth of the watercourse - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P403",
    csv_file=csv_file
)
obj_qs = df['object_q'].dropna().unique().tolist()
if obj_qs:
    vals = ' '.join(f"wd:{q}" for q in obj_qs)
    sparql.setQuery(f"""
    SELECT ?item ?coord WHERE {{
      VALUES ?item {{ {vals} }}
      ?item wdt:P625 ?coord .
    }}
    """)
    res = sparql.query().convert()

    point_map = {}
    dms_map = {}
    for b in res['results']['bindings']:
        q = b['item']['value'].rsplit('/', 1)[-1]
        wkt = b['coord']['value']  # e.g. Point(-7.06818 52.48924)
        point_map[q] = wkt
        match = re.match(r'Point\(([-\d.]+) ([-\d.]+)\)', wkt)
        if match:
            lon, lat = map(float, match.groups())
            dms_map[q] = dms_string(lat, lon)
else:
    point_map = {}
    dms_map = {}

# Append both formats to the dataframe
df['object-point'] = df['object_q'].map(point_map)
df['object-coords'] = df['object_q'].map(dms_map)

# Save result
df.to_csv("mouth_of_watercourse.csv", index=False)

#### IDS

In [ ]:
#all properties that are instance of something that is subclass of unique id(Q6545185) - modelled in protege
# returns a bit too much weird check again.
def get_ids(target_qcode):
    query = f"""
    SELECT ?prop ?propLabel ?value WHERE {{
      wd:{target_qcode} ?p ?value.

      FILTER(STRSTARTS(STR(?p), STR(wdt:)))

      BIND(IRI(REPLACE(STR(?p), STR(wdt:), STR(wd:))) AS ?prop)

      ?prop wdt:P31 / wdt:P279* wd:Q6545185.

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """

    sparql.setQuery(query)
    results = sparql.query().convert()

    output = []
    for result in results["results"]["bindings"]:
        row = {
            "subject_q": target_qcode,
            "p_label": result.get("propLabel", {}).get("value", ""),
            "predicate": result.get("prop", {}).get("value", "").split("/")[-1],
            "value": result.get("value", {}).get("value", "")
        }
        output.append(row)

    return pd.DataFrame(output, columns=[
        "subject_q", "p_label", "predicate", "value"
    ])


In [6]:
df = get_ids(target_qcode)
df.to_csv("ids.csv", index=False)

#### Others

In [7]:
def extract_entity_relation_info(subject_qcode, predicate_pcode, csv_file):

    # Load data
    triples = pd.read_csv(csv_file)
    q_labels_df = pd.read_csv(qcode_labels_file)
    q_dict = dict(zip(q_labels_df['Q-code'], q_labels_df['Label']))

    p_labels_df = pd.read_csv(pcode_labels_file)
    p_dict = dict(zip(p_labels_df['P-code'], p_labels_df['Label']))
    p_label = p_dict.get(predicate_pcode, predicate_pcode)

    # URI helpers
    def uri_for_q(q): return f"http://www.wikidata.org/entity/{q}"
    def uri_for_p(p): return f"http://www.wikidata.org/prop/direct/{p}"
    def extract_q(uri): return re.search(r'Q\d+', uri).group(0) if isinstance(uri, str) and re.search(r'Q\d+', uri) else None

    # SPARQL for instance of (P31)
    def get_instance_of(qcode):
        sparql.setQuery(f"""
        SELECT ?type WHERE {{
          wd:{qcode} wdt:P31 ?type .
        }}
        """)
        try:
            results = sparql.query().convert()
            return [extract_q(result["type"]["value"]) for result in results["results"]["bindings"]]
        except:
            return []

    # Extract matching triples
    subject_uri = uri_for_q(subject_qcode)
    predicate_uri = uri_for_p(predicate_pcode)
    links = triples[
        (triples["subject"] == subject_uri) &
        (triples["predicate"] == predicate_uri)
    ].copy()
    links['object_q'] = links['object'].apply(extract_q)
    links = links.dropna(subset=['object_q'])

    # Get instance_of Q-codes
    instance_data = []
    for q in links['object_q'].unique():
        p31_targets = get_instance_of(q)
        if not p31_targets:
            instance_data.append((q, ""))
        else:
            for inst in p31_targets:
                instance_data.append((q, inst))

    # Merge instance_of
    instance_df = pd.DataFrame(instance_data, columns=["object_q", "instance_of_q"])
    links = links.merge(instance_df, on='object_q', how='left')

    # Fetch missing Q-code labels
    all_qcodes = set([subject_qcode]) | set(links['object_q']) | set(links['instance_of_q'].dropna())
    missing_qcodes = [q for q in all_qcodes if q not in q_dict]

    if missing_qcodes:
        new_labels = get_labels(missing_qcodes)
        q_dict.update(new_labels)

    # Add label columns
    links['subject_q'] = subject_qcode
    links['subject'] = q_dict.get(subject_qcode, subject_qcode)
    links['object'] = links['object_q'].apply(lambda q: q_dict.get(q, q))
    links["instance_of"] = links['instance_of_q'].apply(lambda q: q_dict.get(q, q) if pd.notnull(q) and q != "" else "")

    # Save updated labels
    q_dict_cleaned = {
        str(k): str(v)
        for k, v in q_dict.items()
        if isinstance(k, str) and isinstance(v, str) and re.match(r"Q\d+", str(k))
    }
    updated_df = pd.DataFrame(sorted(q_dict_cleaned.items()), columns=["Q-code", "Label"])
    final_df = links[['subject_q', 'subject', 'object_q', 'object', 'instance_of_q', "instance_of"]]
    output_filename = f"{p_label}.csv"

    return final_df


In [ ]:
#described by source - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P1343",
    csv_file=csv_file
)
df.to_csv("described_by_source.csv", index=False)

In [ ]:
# located in the administrative territorial entity - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P131",
    csv_file=csv_file
)
df.to_csv(df, "located_in_the_administrative_territorial_entity.csv", index=False)

In [ ]:
#named after - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P138",
    csv_file=csv_file
)
df.to_csv("named_after.csv", index=False)

In [ ]:
#location - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P276",
    csv_file=csv_file
)
df.to_csv("location.csv", index=False)

In [ ]:
#diocese - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P708",
    csv_file=csv_file
)
df.to_csv("diocese.csv", index=False)

In [ ]:
#feast day - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P841",
    csv_file=csv_file
)
df.to_csv("feast_day.csv", index=False)
#TODO if RRULE is ok, change Month name to month number

In [ ]:
#country - modeled in protege
df = extract_entity_relation_info(
    subject_qcode=target_qcode,
    predicate_pcode="P17",
    csv_file=csv_file
)
df.to_csv("country.csv", index=False)

In [ ]:
#historic county - Lowrys Well - model with falls within. are there any time constraints?
df = extract_entity_relation_info(
    subject_qcode="Q131625906",
    predicate_pcode="P7959",
    csv_file=csv_file
)
df.to_csv("historic_county.csv", index=False)

In [ ]:
# medical condition treated - modeled in protege with P15i influenced
df = extract_entity_relation_info(
    subject_qcode="Q126472950",
    predicate_pcode="P2175",
    csv_file=csv_file
)
df.to_csv("medical_treated.csv", index=False)


In [ ]:
#medical condition - modeled in protege with P15i influenced
df = extract_entity_relation_info(
    subject_qcode="Q122302608",
    predicate_pcode="P1050",
    csv_file=csv_file
)
df.to_csv("medical_condition.csv", index=False)

In [ ]:
#significant person - example on st lachtain  - modeled in protege with p15i influenced
df = extract_entity_relation_info(
    subject_qcode="Q121840779",
    predicate_pcode="P3342",
    csv_file=csv_file
)
df.to_csv("significant_person.csv", index=False)


In [ ]:
#patron saint - example on Hermit well - modelled in protege
df = extract_entity_relation_info(
    subject_qcode="Q126478185",
    predicate_pcode="P417",
    csv_file=csv_file
)
df.to_csv("patron_saint.csv", index=False)

In [9]:
#use state as direct properties -- ST Mogues - modeled in protege with P44
df = extract_entity_relation_info(
    subject_qcode="Q126454603",
    predicate_pcode="P5817",
    csv_file=csv_file
)
df.to_csv("use_state.csv", index=False)

In [ ]:
#conservation state as direct properties -- ST Augustine
df = extract_entity_relation_info(
    subject_qcode="44",
    predicate_pcode="P5816",
    csv_file=csv_file
)
df.to_csv("conservation_state.csv", index=False)


## TODO: script automatising it all for one subject (and then all subjects). each P gets a table named after P-code. 